In [ ]:
!pip install great_tables
!pip install -U scikit-learn

# Imports

In [ ]:
import numpy as np 
import pandas as pd
import os
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import optuna
from category_encoders import OneHotEncoder, MEstimateEncoder, CatBoostEncoder, OrdinalEncoder
from sklearn import set_config
import category_encoders
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold, 
                                     TunedThresholdClassifierCV)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score, accuracy_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer,StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.preprocessing import PolynomialFeatures
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import auc, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
import warnings
from great_tables import GT, style ,exibble, from_column, loc
from colorama import Style, Fore


sns.set_theme(style = 'white', palette = 'colorblind')
pal = sns.color_palette('colorblind')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None
warnings.simplefilter(action='ignore', category=FutureWarning)

pd.set_option('display.max_rows', 150)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Functions

In [ ]:
def printColor(pText: str):
    print(f'{Style.BRIGHT}{Fore.GREEN}{pText}{Style.RESET_ALL}')    

In [ ]:
def printInfo():
    print(f'{Style.BRIGHT}{Fore.YELLOW}SHAPE{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.shape}')
    print(f'{Style.BRIGHT}{Fore.GREEN} test:  {test.shape}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original:  {original.shape}')    
    print(f'{Style.BRIGHT}{Fore.YELLOW}\nNULL VALUES{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.isnull().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} test: {test.isnull().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original: {original.isnull().any().any()}')    
    print(f'{Style.BRIGHT}{Fore.YELLOW}\nDUPLICATES{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.duplicated().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} test: {test.duplicated().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original: {original.duplicated().any().any()}')    

In [ ]:
def customStatistic(df: pd.DataFrame(), categoric = False):
    num_cols = list(df._get_numeric_data())
    cat_cols = list(df.drop(num_cols,axis=1))
    if categoric:
        desc = pd.DataFrame(index = list(df[cat_cols]))
        df = df[cat_cols]
    else:
        desc = pd.DataFrame(index = list(df[num_cols]))
        df = df[num_cols]
        desc['skew'] = df[num_cols].skew()
        
    desc['type'] = df.dtypes
    desc['count'] = df.count()
    desc['nunique'] = df.nunique()
    desc['%unique'] = desc['nunique'] /len(df) * 100 
    desc['null'] = df.isnull().sum()
    desc['%null'] = desc['null'] / len(df) * 100
    desc = pd.concat([desc,df.describe().T.drop('count',axis=1)],axis=1)    

    desc = desc.round(2)
    return desc.reset_index().rename(columns={'index':'Column'}).sort_values(by=['type'])

In [ ]:
def min_max_unique(data_train, data_test):
    
    df = pd.DataFrame(index=data_train.columns)
    summary = {}
    for col in data_train.columns:
        if pd.api.types.is_numeric_dtype(data_train[col]):  # Verifica se a coluna é numérica
            min_train = min(data_train[col])
            min_test = min(data_test[col])
            max_train = max(data_train[col])
            max_test = max(data_test[col])
            unique_train = len(data_train[col].unique())
            unique_test = len(data_test[col].unique())
            top5_train = sorted(data_train[col])[:5]
            top5_test = sorted(data_test[col])[:5]
        else:  
            min_train = min_test = max_train = max_test = None
            unique_train = len(data_train[col].unique())
            unique_test = len(data_test[col].unique())
            top5_train = top5_test = None
        summary[col] = [min_train, min_test, max_train, max_test, 
                        unique_train, unique_test]

    df = pd.DataFrame.from_dict(summary, orient='index', columns=['min_train', 'min_test', 'max_train', 'max_test', 
                                                                  'unique_train', 'unique_test'])\
        .reset_index().rename(columns={'index': 'columns'})


    return df

# Data

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s4e6/train.csv', index_col='id')
test = pd.read_csv(r'/kaggle/input/playground-series-s4e6/test.csv', index_col='id')
original = pd.read_csv('/kaggle/input/predict-students-dropout-and-academic-success/data.csv')
sub  = pd.read_csv(r'/kaggle/input/playground-series-s4e6/sample_submission.csv')

In [ ]:
train.head(3)

In [ ]:
test.head(3)

In [ ]:
original.head(3)

In [ ]:
printInfo()

In [ ]:
TARGET = 'Target'
CAT_COLS = ['Marital status', 'Application mode', 'Course',
            'Previous qualification', 'Nacionality', "Mother's qualification", 
            "Father's qualification", "Mother's occupation",
            "Father's occupation"]
NUMERIC_COLS = [f for f in train._get_numeric_data() if (f not in TARGET) and (f not in CAT_COLS)]
print(f'Numeric cols: {len(NUMERIC_COLS)}')
print(f'Cat cols: {len(CAT_COLS)}')

In [ ]:
for feature in CAT_COLS:
    for df in [train, test]:
        df[feature] = df[feature].astype('category')

# Descriptive Statistics

In [ ]:
stat = customStatistic(train,False)
GT(stat)\
    .tab_header(title='Descriptive Statistic - Train', subtitle='Numeric Fields')\
    .data_color(columns=['min','max','mean'],palette=['lightblue','lightcoral'],alpha=0.5)\
    .fmt_percent(columns=['%unique','%null'])

In [ ]:
stat = customStatistic(test,False)
GT(stat)\
    .tab_header(title='Descriptive Statistic - Test', subtitle='Numeric Fields')\
    .data_color(columns=['min','max','mean'],palette=['lightblue','lightcoral'],alpha=0.5)\
    .fmt_percent(columns=['%unique','%null'])

In [ ]:
s = min_max_unique(train.drop(TARGET,axis=1),test)
GT(s)\
    .tab_header(title='Min Max Uniques', subtitle='Train and Test datasets')\
    .data_color(columns=['columns'],palette=['lightgray','lightgray'])    


# Correlation

In [ ]:
def plot_correlation(df,label=''):
    corr = df._get_numeric_data().corr(method='spearman')
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)]=True
    fig, ax = plt.subplots(figsize=(27,20))
    sns.heatmap(data=corr, 
                mask=mask , 
                annot=True,
                cmap='icefire',
                annot_kws={'size': 12, 'rotation': 45},
                ax=ax
                );
    ax.set_title(f'Correlation {label}',fontsize=25, fontweight='bold');

In [ ]:
plot_correlation(train,'train')

# EDA (Numeric Fields)

In [ ]:
def plot_numerical():
    #num = train.select_dtypes(include=['int64','float64']).columns

    df = pd.concat([train[NUMERIC_COLS].assign(Source = 'Train'), 
                    test[NUMERIC_COLS].assign(Source = 'Test')], ignore_index = True)

    # Use of more advanced artistic matplotlib interface (see the axes)
    fig, axes = plt.subplots(len(NUMERIC_COLS), 3 ,figsize = (16, len(NUMERIC_COLS) * 4), 
                             gridspec_kw = {'hspace': 0.35, 'wspace': 0.3, 
                                            'width_ratios': [0.80, 0.20, 0.20]})

    for i,col in enumerate(NUMERIC_COLS):
        ax = axes[i,0]
        sns.kdeplot(data = df[[col, 'Source']], x = col, hue = 'Source', palette=['#456cf0', '#ed7647'], linewidth = 2.1, warn_singular=False, ax = ax) # Use of seaborn with artistic interface
        ax.set_title(f"\n{col}",fontsize = 9)
        ax.grid(visible=True, which = 'both', linestyle = '--', color='lightgrey', linewidth = 0.75)
        ax.set(xlabel = '', ylabel = '')

        ax = axes[i,1]
        sns.boxplot(data = df.loc[df.Source == 'Train', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#456cf0', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Train", fontsize = 9)

        ax = axes[i,2]
        sns.boxplot(data = df.loc[df.Source == 'Test', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#ed7647', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Test", fontsize = 9)

    plt.suptitle(f'\nDistribution analysis - numerical features',fontsize = 12, y = 0.89, x = 0.57, fontweight='bold')
    plt.show()

In [ ]:
plot_numerical()

# Categorical

In [ ]:
def plot_cat(limit_unique=20):
    selectcols = train[CAT_COLS].nunique()<=limit_unique
    cols_ = selectcols[selectcols].index.to_list()
    n_cols = len(cols_)
    fig, ax = plt.subplots(n_cols, 2, figsize=(12, 4 * n_cols))
    for i, coluna in enumerate(cols_):    
        sns.countplot(x=train[coluna], ax=ax[i, 0])
        ax[i, 0].set_title(f'{coluna}')
        ax[i, 0].set_ylabel('Count')
        ax[i, 0].set_xlabel(coluna)
        ax[i, 0].tick_params(axis='x', labelrotation=45)

        for container in ax[i, 0].containers:
            ax[i, 0].bar_label(container, fmt='%d', label_type='center', rotation=90)
            

        s1 = train[coluna].value_counts()        

        textprops = {
            'size':8, 
            'weight': 'bold', 
            'color':'white'
        }

        ax[i, 1].pie(s1,
            autopct='%1.f%%',
            pctdistance=0.8, 
            textprops=textprops,
            labels=train[coluna].value_counts().index
        )    
        ax[i, 1].set_title(f'% {coluna}')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_cat()

# EDA (Target)

In [ ]:
ax = sns.countplot(x=train[TARGET])

total = len(train[TARGET])
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width() / 2., height + 0.1,
            '{:.2%}'.format(height / total),
            ha="center")
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45)
plt.title('Percentage/number of classes',fontweight='bold')
plt.show()

# Cross Validation

## Configs

In [ ]:
SUBMIT=False
SEED = 42
USE_ORIGINAL=False

In [ ]:
scores, oof, test_predictions = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
lblTarget = LabelEncoder()
train[TARGET] = lblTarget.fit_transform(train[TARGET])
kf = StratifiedKFold(n_splits=5,random_state=SEED,shuffle=True)
oof_proba = np.zeros((len(train),3))

In [ ]:
def score_model(estimator, label = '', verbose=False):
    
    X = train.copy()
    y = X.pop(TARGET)
    
    val_predictions = np.zeros((len(X)))
    test_predictions = np.zeros((len(test)))
    val_scores= []  
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
                    
        
        model = clone(estimator)        
        if isinstance(model, CatBoostClassifier):
            model.set_params(cat_features=CAT_COLS)
        
        X_train = X.iloc[train_idx].reset_index(drop = True)
        y_train = y.iloc[train_idx].reset_index(drop = True)

        X_val = X.iloc[val_idx].reset_index(drop = True)
        y_val = y.iloc[val_idx].reset_index(drop = True)
        
        if USE_ORIGINAL:
            X_train = pd.concat([X_train, original.drop([TARGET],axis=1)])
            y_train = pd.concat([y_train,original[TARGET]])
            
        model.fit(X_train, y_train)      
        val_preds = model.predict(X_val)
        val_predictions[val_idx] += val_preds.ravel()      
        oof_proba[val_idx] += model.predict_proba(X_val)                 
        val_score = accuracy_score(y_val, val_preds)
        val_scores.append(val_score)
        if verbose:
            print(f'Fold {fold+1}: {val_score:.5f}')
        
    if SUBMIT:
        X_train = train.copy()
        y_train = X_train.pop(TARGET) 
        model.fit(X_train,y_train)            
        test_predictions = model.predict_proba(test)
        
    printColor(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | {label}')
         

    return val_scores, val_predictions, test_predictions

# Models

In [ ]:
models = [               
    ('cb',CatBoostClassifier(random_state=SEED,
                             cat_features=CAT_COLS,
                             verbose=0)),                                         
    ('rf', RandomForestClassifier(random_state = SEED)),
    ('xgb', XGBClassifier(random_state = SEED)),
    ('lgb', LGBMClassifier(random_state = SEED, verbose=-1)),
    ('dart', LGBMClassifier(random_state = SEED,verbose=-1, boosting_type = 'dart')),    
    ('gb', GradientBoostingClassifier(random_state = SEED)),
    ('hgb', HistGradientBoostingClassifier(random_state = SEED))
]

for (label, model) in models:
    if isinstance(model,CatBoostClassifier):
        scores[label], oof[label], test_predictions[label] = score_model(
                model,
                label)
    else:
        scores[label], oof[label], test_predictions[label] = score_model(
            make_pipeline(ce.MEstimateEncoder(cols=CAT_COLS), model),
            label
    )

# Ensemble

In [ ]:
y_pred = oof.mode(axis=1)[0].astype(int)
scores['ensemble'] = accuracy_score(train[TARGET],y_pred)

# Scores

In [ ]:
ax = scores.mean().sort_values(ascending=True).plot(kind='barh', figsize=(10, 6), color='#a2a28f')

for container in ax.containers:
    ax.bar_label(container, label_type='center', color='black', fontsize=12, fontweight='bold')
ax.patches[-1].set_facecolor('#6ca957')

ax.set_title('Score Models', fontsize=16, fontweight='bold')

ax.set_xlabel('Accuracy', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')

ax.tick_params(axis='both', which='major', labelsize=12)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()